# KeelNet Stage 1 Experiment in Colab

This notebook runs the Stage 1 A/B experiment from `../README.md`:

- Run A: answer-only baseline
- Run B: abstain-aware model
- same backbone, same preprocessing, same training budget
- compare unsupported-answer rate, answerable EM/F1, and abstain metrics


## Before You Run

Use a GPU runtime in Colab.

This notebook assumes you are running it from the repository root after cloning or uploading the repo into the Colab session.

If you only want a smoke test first, set `MAX_TRAIN_SAMPLES` and `MAX_EVAL_SAMPLES` to small values in the config cell below.


In [ ]:
%pip install -q -e .

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import torch

REPO_DIR = Path.cwd()
OUTPUT_ROOT = REPO_DIR / "outputs" / "stage1_colab"
BASELINE_DIR = OUTPUT_ROOT / "baseline"
ABSTAIN_DIR = OUTPUT_ROOT / "abstain"
BASELINE_EVAL = OUTPUT_ROOT / "baseline_eval.json"
ABSTAIN_EVAL = OUTPUT_ROOT / "abstain_eval.json"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "distilbert-base-uncased"
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
MAX_TRAIN_SAMPLES = None  # e.g. 2000 for a quick smoke run
MAX_EVAL_SAMPLES = None   # e.g. 1000 for a quick smoke run

print(f"Repo dir: {REPO_DIR}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def run(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


In [ ]:
run([sys.executable, "-m", "unittest", "discover", "-s", "tests"])

In [ ]:
def train_run(mode: str, output_dir: Path):
    cmd = [
        sys.executable,
        "-m",
        "keelnet.train",
        "--mode",
        mode,
        "--output-dir",
        str(output_dir),
        "--model-name",
        MODEL_NAME,
        "--num-train-epochs",
        str(NUM_EPOCHS),
        "--train-batch-size",
        str(TRAIN_BATCH_SIZE),
        "--eval-batch-size",
        str(EVAL_BATCH_SIZE),
    ]
    if MAX_TRAIN_SAMPLES is not None:
        cmd.extend(["--max-train-samples", str(MAX_TRAIN_SAMPLES)])
    if MAX_EVAL_SAMPLES is not None:
        cmd.extend(["--max-eval-samples", str(MAX_EVAL_SAMPLES)])
    run(cmd)


train_run("baseline", BASELINE_DIR)
train_run("abstain", ABSTAIN_DIR)

In [ ]:
def evaluate_run(mode: str, model_dir: Path, output_path: Path):
    cmd = [
        sys.executable,
        "-m",
        "keelnet.evaluate",
        "--mode",
        mode,
        "--model-path",
        str(model_dir),
        "--output-path",
        str(output_path),
        "--eval-batch-size",
        str(EVAL_BATCH_SIZE),
    ]
    if MAX_EVAL_SAMPLES is not None:
        cmd.extend(["--max-eval-samples", str(MAX_EVAL_SAMPLES)])
    run(cmd)


evaluate_run("baseline", BASELINE_DIR, BASELINE_EVAL)
evaluate_run("abstain", ABSTAIN_DIR, ABSTAIN_EVAL)

In [ ]:
baseline_results = json.loads(BASELINE_EVAL.read_text())
abstain_results = json.loads(ABSTAIN_EVAL.read_text())

comparison = pd.DataFrame(
    [
        {"model": "Run A", **baseline_results["dev_metrics"]},
        {"model": "Run B", **abstain_results["dev_metrics"]},
    ]
)
comparison = comparison[
    [
        "model",
        "answerable_em",
        "answerable_f1",
        "unsupported_answer_rate",
        "abstain_precision",
        "abstain_recall",
        "abstain_f1",
        "overall_em",
        "overall_f1",
    ]
]
comparison.round(2)

In [ ]:
print("Selected abstain threshold:", abstain_results["selected_threshold"])
print("\nValidation metrics for Run B:")
pd.Series(abstain_results["validation_metrics"]).round(2)

## What To Check Next

A win is not just a lower unsupported-answer rate.

Check all three:

- unsupported-answer rate goes down
- answerable F1 does not collapse
- the abstain-aware model is not simply over-abstaining

After that, inspect failure cases using the saved `*_eval.json` files under `outputs/stage1_colab/`.
